In [8]:
from mobslim import Networks, Schedules, Vehicles
from mobslim.animate import animate_events
from mobslim.entities.agents import load_plans_from_xml
from mobslim.expected import SimpleExpectedDurations
from mobslim.listener import EventListener
from mobslim.optimizer import Optimizer
from mobslim.planners.greedy_trip_planner import GreedyTripPlanner
from mobslim.planners.rerouters.simple_rerouter import StaticRouter
from mobslim.sim import Sim

In [9]:
# network setup
networks = Networks()
networks.load_xml("../scenarios/pt-simple/network.xml")
networks._networks["pt"].edges

OutEdgeView([('1', '2'), ('2', '3'), ('3', '4'), ('4', '5'), ('5', '6')])

In [10]:
# agent setup
plans = load_plans_from_xml(
    "../scenarios/pt-simple/population.xml", networks=networks
)
# set mode to pt
for plan in plans.values():
    plan.set_network_mode("pt")

print(plans)

{'1': Plan([SOS(), Act(ActivityType.HOME, loc=1, dur=18060), Trip(pt, 1>5, duration=None, route=None), Act(ActivityType.HOME, loc=5, dur=45300)])}


In [13]:
# transit schedule and vehicles
vehicles = Vehicles()
vehicles.load_xml("../scenarios/pt-simple/transitVehicles.xml")

schedules = Schedules(vehicles=vehicles)
schedules.load_xml("../scenarios/pt-simple/transitSchedule.xml")

schedules.lines["gelb"]["gelb_1"]

{'transport_mode': 'pt',
 'route_profile': [{'ref_id': '192',
   'departure_offset': '00:00:00',
   'arrival_offset': '00:00:00',
   'await_departure': True},
  {'ref_id': '293',
   'departure_offset': '00:00:30',
   'arrival_offset': '00:00:30',
   'await_departure': True},
  {'ref_id': '495',
   'departure_offset': '00:30:00',
   'arrival_offset': '00:30:00',
   'await_departure': True},
  {'ref_id': '596',
   'departure_offset': '00:30:30',
   'arrival_offset': '00:30:30',
   'await_departure': True}],
 'route_links': ['1-2', '2-3', '3-4', '4-5', '5-6'],
 'departures': [{'id': '1000',
   'departure_time': '05:05:00',
   'vehicle_ref_id': '1000'}]}

In [ ]:
# planner setup
expected_link_durations = SimpleExpectedDurations(networks, mode="pt")
router = StaticRouter(
    network=networks, network_mode="pt", expectations=expected_link_durations
)
planner = GreedyTripPlanner(plans=plans, router=router, network=networks, p=0.2)

# initiate all plans with naive trip estimates
planner.plan()

# simulation setup
sim = Sim(networks=networks, listener=EventListener())

# optimizer setup and run
optimizer = Optimizer(
    sim=sim, plans=plans, planner=planner, network_modes=["pt"]
)
events = optimizer.run(max_runs=20)

In [ ]:
start, limit, step = 21500, 22600, 100
ani = animate_events(networks, events, start=start, limit=limit, step=step)
ani